In [5]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta
import numpy as np
from loguru import logger
import os

logger.add("etl_proces.log", rotation="5 MB")


2

 # Stap 1 E(Extract)

In [6]:
bron_db_pad = '../../week_3/DE/sdm.db'
if not os.path.exists(bron_db_pad):
    logger.warning(f"Bron database niet gevonden op pad: {bron_db_pad}. Controleer het pad!")

logger.info(f"Stap 1: Extract - Verbinden met bron database '{bron_db_pad}'")
bron_conn = sqlite3.connect(bron_db_pad)


2026-04-09 14:43:50.118 | INFO     | __main__:<module>:5 - Stap 1: Extract - Verbinden met bron database '../../week_3/DE/sdm.db'


In [7]:
logger.info("Ophalen Inkoop data...")
df_acc_inkoop = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", bron_conn)
df_acc_inkoop_acc = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop_Accessoire", bron_conn)
df_acc_inkoop_lev = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop_Leverancier", bron_conn)

df_fiets_inkoop = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", bron_conn)
df_fiets_inkoop_fab = pd.read_sql_query("SELECT * FROM Fiets_Inkoop_Fabrikant", bron_conn)
df_fiets_inkoop_fiets = pd.read_sql_query("SELECT * FROM Fiets_Inkoop_Fiets", bron_conn)


2026-04-09 14:43:50.128 | INFO     | __main__:<module>:1 - Ophalen Inkoop data...


In [8]:
logger.info("Ophalen Verkoop data...")
df_acc_verk = pd.read_sql_query("SELECT * FROM Accessoireverkoop", bron_conn)
df_acc_verk_acc = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Accessoire", bron_conn)
df_acc_verk_fil = pd.read_sql_query("SELECT * FROM Accessoireverkoop_filiaal", bron_conn)
df_acc_verk_klant = pd.read_sql_query("SELECT * FROM Accessoireverkoop_klant", bron_conn)
df_acc_verk_lev = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Leverancier", bron_conn)
df_acc_verk_mont = pd.read_sql_query("SELECT * FROM Accessoireverkoop_Monteur", bron_conn)

df_fiets_verk = pd.read_sql_query("SELECT * FROM Fietsverkoop", bron_conn)
df_fiets_verk_fab = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fabrikant", bron_conn)
df_fiets_verk_fiets = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fiets", bron_conn)
df_fiets_verk_fil = pd.read_sql_query("SELECT * FROM Fietsverkoop_filiaal", bron_conn)
df_fiets_verk_klant = pd.read_sql_query("SELECT * FROM Fietsverkoop_klant", bron_conn)
df_fiets_verk_mont = pd.read_sql_query("SELECT * FROM Fietsverkoop_Monteur", bron_conn)


2026-04-09 14:43:50.144 | INFO     | __main__:<module>:1 - Ophalen Verkoop data...


In [9]:
logger.info("Ophalen Onderhoud data...")
df_ond = pd.read_sql_query("SELECT * FROM Onderhoud", bron_conn)
df_ond_fab = pd.read_sql_query("SELECT * FROM Onderhoud_Fabrikant", bron_conn)
df_ond_fiets = pd.read_sql_query("SELECT * FROM Onderhoud_Fiets", bron_conn)
df_ond_fil = pd.read_sql_query("SELECT * FROM Onderhoud_Filiaal", bron_conn)
df_ond_mont = pd.read_sql_query("SELECT * FROM Onderhoud_Monteur", bron_conn)


2026-04-09 14:43:50.161 | INFO     | __main__:<module>:1 - Ophalen Onderhoud data...


In [10]:
bron_conn.close()
logger.success("Stap 1: Extract afgerond. Data is ingeladen in werkgeheugen.")


2026-04-09 14:43:50.174 | SUCCESS  | __main__:<module>:2 - Stap 1: Extract afgerond. Data is ingeladen in werkgeheugen.


 # Stap 2 T (Transform)

In [11]:
logger.info("Stap 2: Transform - Bouwen van Dimensies en Feiten...")

# DIM KLANT
dim_klant = pd.concat([df_acc_verk_klant, df_fiets_verk_klant]).drop_duplicates(
    subset=['klantnr']).reset_index(drop=True)
huidig_jaar = datetime.now().year
dim_klant['geboortedatum'] = pd.to_datetime(
    dim_klant['geboortedatum'], errors='coerce')
dim_klant['leeftijd'] = huidig_jaar - dim_klant['geboortedatum'].dt.year
dim_klant['leeftijdgroep'] = pd.cut(dim_klant['leeftijd'], bins=[
                                    0, 20, 40, 60, 100], labels=['<20', '20-40', '40-60', '60+'])
dim_klant = dim_klant.drop(columns=['leeftijd'])
logger.info(f"-> DIM_KLANT getransformeerd: {len(dim_klant)} unieke klanten.")


2026-04-09 14:43:50.197 | INFO     | __main__:<module>:1 - Stap 2: Transform - Bouwen van Dimensies en Feiten...
2026-04-09 14:43:50.205 | INFO     | __main__:<module>:13 - -> DIM_KLANT getransformeerd: 25 unieke klanten.


In [12]:
# DIM FIETS
dim_fiets_basis = pd.concat([df_fiets_inkoop_fiets, df_fiets_verk_fiets,
                            df_ond_fiets]).drop_duplicates(subset=['fietsnr'])
dim_fabrikant = pd.concat([df_fiets_inkoop_fab, df_fiets_verk_fab,
                          df_ond_fab]).drop_duplicates(subset=['fabrikantnr'])
# Koppel fiets aan fabrikant
dim_fiets = pd.merge(dim_fiets_basis, dim_fabrikant, left_on='fabrikant',
                     right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))
dim_fiets = dim_fiets.rename(columns={
                             'naam': 'fabrikantnaam', 'adres': 'fabrikant_adres', 'plaats': 'fabrikant_plaats'})
dim_fiets['prijsklasse'] = np.where(
    dim_fiets['standaardprijs'] > 1000, 'Duur Segment', 'Basis Segment')
logger.info(f"-> DIM_FIETS getransformeerd: {len(dim_fiets)} unieke fietsen.")


2026-04-09 14:43:50.220 | INFO     | __main__:<module>:13 - -> DIM_FIETS getransformeerd: 75 unieke fietsen.


In [13]:
# DIM ACCESSOIRE
dim_acc_basis = pd.concat(
    [df_acc_inkoop_acc, df_acc_verk_acc]).drop_duplicates(subset=['accessoirenr'])
dim_lev = pd.concat([df_acc_inkoop_lev, df_acc_verk_lev]
                    ).drop_duplicates(subset=['leveranciernr'])
# Koppel accessoire aan leverancier
dim_accessoire = pd.merge(dim_acc_basis, dim_lev, left_on='leverancier',
                          right_on='leveranciernr', how='left', suffixes=('', '_leverancier'))
dim_accessoire = dim_accessoire.rename(
    columns={'naam_leverancier': 'leveranciernaam', 'adres': 'leverancier_adres', 'woonplaats': 'leverancier_woonplaats'})
logger.info(f"-> DIM_ACCESSOIRE getransformeerd: {len(dim_accessoire)} unieke accessoires.")


2026-04-09 14:43:50.236 | INFO     | __main__:<module>:11 - -> DIM_ACCESSOIRE getransformeerd: 10 unieke accessoires.


In [14]:
# DIM MONTEUR
dim_monteur_basis = pd.concat(
    [df_acc_verk_mont, df_fiets_verk_mont, df_ond_mont]).drop_duplicates(subset=['monteurnr'])
dim_filiaal = pd.concat([df_acc_verk_fil, df_fiets_verk_fil,
                        df_ond_fil]).drop_duplicates(subset=['filiaalnr'])
# Koppel monteur aan filiaal
dim_monteur = pd.merge(dim_monteur_basis, dim_filiaal, left_on='filiaal',
                       right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))
dim_monteur = dim_monteur.rename(
    columns={'naam_filiaal': 'filiaalnaam', 'adres': 'filiaal_adres'})
dim_monteur['salarisgroep'] = np.where(
    dim_monteur['uurloon'] > 20, 'Senior', 'Medior')
logger.info(f"-> DIM_MONTEUR getransformeerd: {len(dim_monteur)} unieke monteurs.")


2026-04-09 14:43:50.261 | INFO     | __main__:<module>:13 - -> DIM_MONTEUR getransformeerd: 15 unieke monteurs.


In [15]:
# DIM DATUM
dim_datum = pd.DataFrame(
    {'datum': pd.date_range(start='2020-01-01', end='2025-12-31')})
dim_datum['datum_id'] = dim_datum['datum'].dt.strftime('%Y%m%d').astype(int)
dim_datum['dag'] = dim_datum['datum'].dt.day
dim_datum['maand'] = dim_datum['datum'].dt.month
dim_datum['jaar'] = dim_datum['datum'].dt.year
dim_datum['maandnaam'] = dim_datum['datum'].dt.strftime('%B')
dim_datum['maand_id'] = dim_datum['datum'].dt.strftime('%Y%m').astype(int)
logger.info(f"-> DIM_DATUM getransformeerd: {len(dim_datum)} datums gegenereerd.")


2026-04-09 14:43:50.353 | INFO     | __main__:<module>:10 - -> DIM_DATUM getransformeerd: 2192 datums gegenereerd.


In [16]:
# FEIT INKOOP
df_acc_inkoop_feit = df_acc_inkoop.copy()
df_acc_inkoop_feit = df_acc_inkoop_feit.rename(
    columns={'accessoire': 'accessoirenummer'})
df_acc_inkoop_feit['fietsnr'] = None  # Heeft geen fiets

df_fiets_inkoop_feit = df_fiets_inkoop.copy()
df_fiets_inkoop_feit = df_fiets_inkoop_feit.rename(
    columns={'fiets': 'fietsnr'})
df_fiets_inkoop_feit['accessoirenummer'] = None  # Heeft geen accessoire

feit_inkoop = pd.concat(
    [df_acc_inkoop_feit, df_fiets_inkoop_feit]).reset_index(drop=True)
feit_inkoop['inkoop_id'] = feit_inkoop.index + 1  # Nieuwe unieke DWH sleutel
# Maak maand_id: 2023 en 10 wordt 202310
feit_inkoop['maand_id'] = feit_inkoop['inkoopjaar'].astype(
    str) + feit_inkoop['inkoopmaand'].astype(str).str.zfill(2)
feit_inkoop['maand_id'] = feit_inkoop['maand_id'].astype(int)


In [17]:
feit_inkoop = pd.merge(
    feit_inkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_inkoop = pd.merge(feit_inkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                       left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_inkoop['inkoopprijs_per_stuk'] = feit_inkoop['inkoopprijs_fiets'].fillna(
    feit_inkoop['inkoopprijs_acc'])
feit_inkoop['totale_inkoopkosten'] = feit_inkoop['aantal'] * \
    feit_inkoop['inkoopprijs_per_stuk']

feit_inkoop = feit_inkoop[['inkoop_id', 'maand_id', 'fietsnr',
                           'accessoirenummer', 'aantal', 'totale_inkoopkosten']]
logger.info(f"-> FEIT_INKOOP getransformeerd: {len(feit_inkoop)} rijen.")


2026-04-09 14:43:50.401 | INFO     | __main__:<module>:12 - -> FEIT_INKOOP getransformeerd: 100 rijen.


In [18]:
# FEIT VERKOOP
df_acc_verk_feit = df_acc_verk.copy()
df_acc_verk_feit = df_acc_verk_feit.rename(
    columns={'accessoire': 'accessoirenummer', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_acc_verk_feit['fietsnr'] = None

df_fiets_verk_feit = df_fiets_verk.copy()
df_fiets_verk_feit = df_fiets_verk_feit.rename(
    columns={'fiets': 'fietsnr', 'klant': 'klantnr', 'monteur': 'monteurnr'})
df_fiets_verk_feit['accessoirenummer'] = None

feit_verkoop = pd.concat(
    [df_acc_verk_feit, df_fiets_verk_feit]).reset_index(drop=True)
feit_verkoop['verkoop_id'] = feit_verkoop.index + 1  # Unieke sleutel
feit_verkoop['datum'] = pd.to_datetime(feit_verkoop['datum'])
feit_verkoop['datum_id'] = feit_verkoop['datum'].dt.strftime(
    '%Y%m%d').astype(int)


In [19]:
feit_verkoop = pd.merge(
    feit_verkoop, dim_fiets[['fietsnr', 'inkoopprijs']], on='fietsnr', how='left')
feit_verkoop = pd.merge(feit_verkoop, dim_accessoire[['accessoirenr', 'inkoopprijs']],
                        left_on='accessoirenummer', right_on='accessoirenr', how='left', suffixes=('_fiets', '_acc'))
feit_verkoop['kostprijs_per_stuk'] = feit_verkoop['inkoopprijs_fiets'].fillna(
    feit_verkoop['inkoopprijs_acc'])
feit_verkoop['omzet'] = feit_verkoop['verkoopprijs']
feit_verkoop['winst'] = feit_verkoop['omzet'] - \
    (feit_verkoop['aantal'] * feit_verkoop['kostprijs_per_stuk'])

feit_verkoop = feit_verkoop[['verkoop_id', 'datum_id', 'klantnr', 'monteurnr',
                             'fietsnr', 'accessoirenummer', 'verkoopprijs', 'omzet', 'winst']]
logger.info(f"-> FEIT_VERKOOP getransformeerd: {len(feit_verkoop)} rijen.")


2026-04-09 14:43:50.453 | INFO     | __main__:<module>:13 - -> FEIT_VERKOOP getransformeerd: 250 rijen.


In [20]:
# FEIT ONDERHOUD
feit_onderhoud = df_ond.copy()
feit_onderhoud = feit_onderhoud.rename(
    columns={'fiets': 'fietsnr', 'monteur': 'monteurnr'})
feit_onderhoud['datum'] = pd.to_datetime(feit_onderhoud['datum'])
feit_onderhoud['datum_id'] = feit_onderhoud['datum'].dt.strftime(
    '%Y%m%d').astype(int)


In [21]:
feit_onderhoud['start_dt'] = pd.to_datetime(
    feit_onderhoud['starttijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['eind_dt'] = pd.to_datetime(
    feit_onderhoud['eindtijd'].astype(str).str[:5], format='%H:%M', errors='coerce')
feit_onderhoud['duur'] = (feit_onderhoud['eind_dt'] -
                          feit_onderhoud['start_dt']).dt.total_seconds() / 3600.0

feit_onderhoud = feit_onderhoud[['onderhoudnr', 'datum_id',
                                 'fietsnr', 'monteurnr', 'starttijd', 'eindtijd', 'duur']]
feit_onderhoud = feit_onderhoud.rename(columns={'onderhoudnr': 'onderhoud_id'})
logger.info(f"-> FEIT_ONDERHOUD getransformeerd: {len(feit_onderhoud)} rijen.")


2026-04-09 14:43:50.491 | INFO     | __main__:<module>:11 - -> FEIT_ONDERHOUD getransformeerd: 50 rijen.


In [22]:
# Datum tabel opruimen
dim_datum = dim_datum.drop(columns=['datum'])
logger.success("Stap 2: Transform afgerond.")


2026-04-09 14:43:50.506 | SUCCESS  | __main__:<module>:3 - Stap 2: Transform afgerond.


 # Stap 3 L (Load)

In [25]:
# %% [markdown]
#  # Stap 3 L (Load) - Gecorrigeerd met SCD-migratie check

# %%
dwh_db = 'fietsen_dwh.db'
logger.info(f"Stap 3: Load - Starten proces in '{dwh_db}'...")

dwh_conn = sqlite3.connect(dwh_db)
cursor = dwh_conn.cursor()

# Zet foreign keys TIJDELIJK UIT om DROP TABLE toe te staan
cursor.execute("PRAGMA foreign_keys = OFF;")

try:
    # --- 1. SCD TYPE 1 FUNCTIE ---
    def load_scd_type_1(df, table_name, business_key, conn):
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        conn.execute(
            f"CREATE UNIQUE INDEX IF NOT EXISTS idx_{table_name} ON {table_name}({business_key});")
        logger.info(f"-> {table_name} (SCD Type 1) geladen.")

    # Laad de Type 1 dimensies
    load_scd_type_1(dim_datum, 'DIM_DATUM', 'datum_id', dwh_conn)
    load_scd_type_1(dim_klant, 'DIM_KLANT', 'klantnr', dwh_conn)
    load_scd_type_1(dim_fiets, 'DIM_FIETS', 'fietsnr', dwh_conn)
    load_scd_type_1(dim_accessoire, 'DIM_ACCESSOIRE', 'accessoirenr', dwh_conn)

    # --- 2. SCD TYPE 2 VOOR DIM_MONTEUR ---
    logger.info("Starten SCD Type 2 voor DIM_MONTEUR...")

    # Check of de tabel bestaat EN of de kolom 'is_current' erin zit
    check_col_query = "PRAGMA table_info(DIM_MONTEUR);"
    cursor.execute(check_col_query)
    columns = [info[1] for info in cursor.fetchall()]

    # Als de tabel niet bestaat OF de SCD2 kolom ontbreekt: Reset de tabel
    if not columns or 'is_current' not in columns:
        logger.warning(
            "DIM_MONTEUR tabel is verouderd of bestaat niet. Tabel wordt (opnieuw) aangemaakt.")
        cursor.execute("DROP TABLE IF EXISTS DIM_MONTEUR;")

        dim_monteur_scd = dim_monteur.copy()
        dim_monteur_scd['valid_from'] = datetime.now().strftime(
            '%Y-%m-%d %H:%M:%S')
        dim_monteur_scd['valid_to'] = '9999-12-31 23:59:59'
        dim_monteur_scd['is_current'] = 1
        dim_monteur_scd.insert(
            0, 'monteur_sk', range(1, len(dim_monteur_scd) + 1))
        dim_monteur_scd.to_sql('DIM_MONTEUR', dwh_conn,
                               if_exists='replace', index=False)
    else:
        # --- DE ECHTE SCD TYPE 2 LOGICA ---
        existing = pd.read_sql_query(
            "SELECT * FROM DIM_MONTEUR WHERE is_current = 1", dwh_conn)

        # Merge bron met huidige DWH data
        merged = pd.merge(dim_monteur, existing, on='monteurnr',
                          how='left', suffixes=('', '_old'))

        # A. Nieuwe monteurs
        new_monteurs_mask = merged['is_current'].isna()
        new_monteurs = merged[new_monteurs_mask].copy()

        # B. Gewijzigde monteurs (bijv. uurloon of filiaal veranderd)
        changed_mask = (
            (merged['is_current'] == 1) &
            ((merged['uurloon'] != merged['uurloon_old']) |
             (merged['filiaalnaam'] != merged['filiaalnaam_old']))
        )
        changed_monteurs = merged[changed_mask].copy()

        now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        max_sk = cursor.execute(
            "SELECT MAX(monteur_sk) FROM DIM_MONTEUR").fetchone()[0] or 0

        # Verwerk wijzigingen
        if not changed_monteurs.empty:
            for _, row in changed_monteurs.iterrows():
                # Oude record sluiten
                cursor.execute("UPDATE DIM_MONTEUR SET is_current = 0, valid_to = ? WHERE monteurnr = ? AND is_current = 1",
                               (now_str, row['monteurnr']))

            # Nieuwe records voorbereiden
            to_insert = changed_monteurs[dim_monteur.columns].copy()
            to_insert['valid_from'] = now_str
            to_insert['valid_to'] = '9999-12-31 23:59:59'
            to_insert['is_current'] = 1
            to_insert.insert(0, 'monteur_sk', range(
                int(max_sk) + 1, int(max_sk) + 1 + len(to_insert)))
            to_insert.to_sql('DIM_MONTEUR', dwh_conn,
                             if_exists='append', index=False)
            max_sk += len(to_insert)

        # Verwerk nieuwe records
        if not new_monteurs.empty:
            to_insert_new = new_monteurs[dim_monteur.columns].copy()
            to_insert_new['valid_from'] = now_str
            to_insert_new['valid_to'] = '9999-12-31 23:59:59'
            to_insert_new['is_current'] = 1
            to_insert_new.insert(0, 'monteur_sk', range(
                int(max_sk) + 1, int(max_sk) + 1 + len(to_insert_new)))
            to_insert_new.to_sql('DIM_MONTEUR', dwh_conn,
                                 if_exists='append', index=False)

    logger.success("Dimensies succesvol geladen.")

    # --- 3. FEITEN KOPPELEN AAN MONTEUR_SK ---
    # Haal de huidige SK's op om de feiten te koppelen
    current_monteurs = pd.read_sql_query(
        "SELECT monteur_sk, monteurnr FROM DIM_MONTEUR WHERE is_current=1", dwh_conn)

    feit_verkoop_sk = pd.merge(feit_verkoop, current_monteurs,
                               on='monteurnr', how='left').drop(columns=['monteurnr'])
    feit_onderhoud_sk = pd.merge(
        feit_onderhoud, current_monteurs, on='monteurnr', how='left').drop(columns=['monteurnr'])

    # --- 4. FEITEN LADEN ---
    # FEIT: VERKOOP
    cursor.execute("DROP TABLE IF EXISTS FEIT_VERKOOP;")
    cursor.execute("""
    CREATE TABLE FEIT_VERKOOP (
        verkoop_id INTEGER PRIMARY KEY,
        datum_id INTEGER,
        klantnr INTEGER,
        monteur_sk INTEGER,
        fietsnr INTEGER,
        accessoirenummer INTEGER,
        verkoopprijs REAL,
        omzet REAL,
        winst REAL,
        FOREIGN KEY(datum_id) REFERENCES DIM_DATUM(datum_id),
        FOREIGN KEY(monteur_sk) REFERENCES DIM_MONTEUR(monteur_sk)
    );
    """)
    feit_verkoop_sk.to_sql('FEIT_VERKOOP', dwh_conn,
                           if_exists='append', index=False)

    # FEIT: ONDERHOUD
    cursor.execute("DROP TABLE IF EXISTS FEIT_ONDERHOUD;")
    cursor.execute("""
    CREATE TABLE FEIT_ONDERHOUD (
        onderhoud_id INTEGER PRIMARY KEY,
        datum_id INTEGER,
        fietsnr INTEGER,
        monteur_sk INTEGER,
        starttijd TEXT,
        eindtijd TEXT,
        duur REAL,
        FOREIGN KEY(datum_id) REFERENCES DIM_DATUM(datum_id),
        FOREIGN KEY(monteur_sk) REFERENCES DIM_MONTEUR(monteur_sk)
    );
    """)
    feit_onderhoud_sk.to_sql('FEIT_ONDERHOUD', dwh_conn,
                             if_exists='append', index=False)

    # FEIT: INKOOP
    feit_inkoop.to_sql('FEIT_INKOOP', dwh_conn,
                       if_exists='replace', index=False)

    dwh_conn.commit()
    logger.success("Stap 3: Load succesvol afgerond.")

except Exception as e:
    dwh_conn.rollback()
    logger.error(f"Fout tijdens laden: {e}")
    raise e

finally:
    cursor.execute("PRAGMA foreign_keys = ON;")
    dwh_conn.close()

2026-04-09 14:47:27.356 | INFO     | __main__:<module>:6 - Stap 3: Load - Starten proces in 'fietsen_dwh.db'...


2026-04-09 14:47:27.385 | INFO     | __main__:load_scd_type_1:20 - -> DIM_DATUM (SCD Type 1) geladen.
2026-04-09 14:47:27.403 | INFO     | __main__:load_scd_type_1:20 - -> DIM_KLANT (SCD Type 1) geladen.
2026-04-09 14:47:27.425 | INFO     | __main__:load_scd_type_1:20 - -> DIM_FIETS (SCD Type 1) geladen.
2026-04-09 14:47:27.444 | INFO     | __main__:load_scd_type_1:20 - -> DIM_ACCESSOIRE (SCD Type 1) geladen.
2026-04-09 14:47:27.446 | INFO     | __main__:<module>:29 - Starten SCD Type 2 voor DIM_MONTEUR...
2026-04-09 14:47:27.446 | WARNING  | __main__:<module>:38 - DIM_MONTEUR tabel is verouderd of bestaat niet. Tabel wordt (opnieuw) aangemaakt.
2026-04-09 14:47:27.463 | SUCCESS  | __main__:<module>:105 - Dimensies succesvol geladen.
2026-04-09 14:47:27.509 | SUCCESS  | __main__:<module>:161 - Stap 3: Load succesvol afgerond.
